# Hermes Agent 101: A Practical Guide to Persistent and Self-Improving Agents

Run from the companion repository root. See the adjacent topic folder README for pinned setup and execution boundaries. Outputs are intentionally not committed.

Cloudflare live requests, Claude sessions/scheduling and Hermes messaging/cron are not part of the default checks.

In [ ]:
from pathlib import Path
import os,sys,tempfile,json,subprocess
folder=Path.cwd()/'hermes-agent-101'
if not folder.exists():folder=Path.cwd()
sys.path.insert(0,str(folder))
from guarded_openai import guarded_network
hermes_root=Path(tempfile.mkdtemp(prefix='tdb-hermes-home-'))
os.environ['HERMES_HOME']=str(hermes_root)
os.environ['HERMES_SKIP_AUTO_UPDATE']='1'
os.environ['HERMES_DISABLE_TELEMETRY']='1'
(hermes_root/'config.yaml').write_text('agent:\n  api_max_retries: 1\n  auto_recovery_cycles: 0\n  environment_probe: false\n  task_completion_guidance: false\n  parallel_tool_call_guidance: false\n  stall_guards: false\n')
from hermes_constants import get_hermes_home
assert get_hermes_home()==hermes_root
print('Isolated HERMES_HOME configured; OS HOME unchanged.')

In [ ]:
from hermes_cli.profiles import create_profile,get_profile_dir
coder=create_profile('coder',no_alias=True,no_skills=True,description='Fictional tutorial coding profile')
researcher=create_profile('researcher',no_alias=True,no_skills=True,description='Fictional tutorial research profile')
assert coder==hermes_root/'profiles/coder' and researcher==hermes_root/'profiles/researcher'
assert coder!=researcher
print('Two real Hermes profiles created with no shell aliases, bundled skills or gateways.')

In [ ]:
os.environ['HERMES_HOME']=str(coder)
from tools.memory_tool import MemoryStore,memory_tool
store=MemoryStore();store.load_from_disk()
answer=json.loads(memory_tool(action='add',target='memory',content='Fictional repository: backend tests run with pnpm test:api.',store=store))
assert answer['success']
reloaded=MemoryStore();reloaded.load_from_disk()
assert reloaded.memory_entries==store.memory_entries
assert not (researcher/'memories/MEMORY.md').exists()
print('Native memory tool writes persist after reload and are absent from the other profile.')

In [ ]:
(coder/'SOUL.md').write_text('# SOUL.md\nYou are a pragmatic engineering assistant. Verify before reporting completion. Never store secrets.\n')
skill_dir=coder/'skills/tutorial-api-check';skill_dir.mkdir(parents=True,exist_ok=True)
(skill_dir/'SKILL.md').write_text('---\nname: tutorial-api-check\ndescription: Use for the fictional tutorial API test workflow.\n---\n## Procedure\nRead the project test command. Run the focused test only with permission. Report actual results.\n')
from tools.skills_tool import skills_list,skill_view
listing=json.loads(skills_list());view=json.loads(skill_view('tutorial-api-check'))
assert 'tutorial-api-check' in json.dumps(listing) and 'Report actual results' in json.dumps(view)
print('Native skill discovery and full skill loading passed; no skill procedure executed.')

In [ ]:
RUN_OPENAI=os.getenv('RUN_OPENAI')=='1'
ledger=Path(os.getenv('TEST_BUDGET_LEDGER',str(hermes_root/'usage.json')))
if RUN_OPENAI:
    assert os.getenv('OPENAI_API_KEY'),'Configure OPENAI_API_KEY locally, never in this notebook.'
    notebook_streams=sys.stdout,sys.stderr
    with guarded_network(ledger):
        from run_agent import AIAgent
        agent=AIAgent(base_url='https://api.openai.com/v1',api_key=os.environ['OPENAI_API_KEY'],provider='openai',api_mode='chat_completions',model='gpt-5.4-mini',max_iterations=3,max_tokens=768,reasoning_config={'effort':'none'},enabled_toolsets=['memory'],skip_context_files=True,skip_memory=True,skip_background_review=True,quiet_mode=True,save_trajectories=False,cwd=str(coder),load_soul_identity=True)
        agent._disable_streaming=True
        try:
            assert [tool['function']['name'] for tool in agent.tools]==['memory']
            result=agent.run_conversation('Use the memory tool to save exactly this confirmed stable fictional convention: Generated files under dist/ must not be edited manually. Then briefly confirm.')
            assert result['completed'] and not result.get('failed')
            assert any(row.get('role')=='tool' and row.get('name')=='memory' and json.loads(row['content']).get('success') for row in result['messages'])
            print(result['final_response'])
        finally:
            agent.close()
            sys.stdout,sys.stderr=notebook_streams
    fresh=MemoryStore();fresh.load_from_disk()
    assert 'dist/' in ' '.join(fresh.memory_entries)
    print('Actual model-selected native memory write persisted.')
else:print('Paid inference disabled; set RUN_OPENAI=1 only with your own approved budget.')

In [ ]:
if RUN_OPENAI:
    notebook_streams=sys.stdout,sys.stderr
    with guarded_network(ledger):
        agent=AIAgent(base_url='https://api.openai.com/v1',api_key=os.environ['OPENAI_API_KEY'],provider='openai',api_mode='chat_completions',model='gpt-5.4-mini',max_iterations=2,max_tokens=512,reasoning_config={'effort':'none'},enabled_toolsets=['memory'],skip_context_files=True,skip_memory=True,skip_background_review=True,quiet_mode=True,save_trajectories=False,cwd=str(coder),load_soul_identity=True)
        agent._disable_streaming=True
        try:
            recalled=agent.run_conversation('From the confirmed fictional project memory, what command runs backend tests, and which generated directory must not be edited? Do not add or change memory. Answer briefly.')
            assert recalled['completed'] and not recalled.get('failed')
            assert 'pnpm test:api' in recalled['final_response'] and 'dist/' in recalled['final_response']
            print(recalled['final_response'])
        finally:
            agent.close()
            sys.stdout,sys.stderr=notebook_streams
    state=json.loads(ledger.read_text())
    assert state['reserved']<=1 and all(row.get('status')==200 for row in state['calls'])
    print('Fresh-agent recall passed; cumulative admitted USD:',round(state['reserved'],6))

In [ ]:
os.environ['HERMES_HOME']=str(researcher)
separate=MemoryStore();separate.load_from_disk()
assert not separate.memory_entries
assert not (coder/'cron/jobs.json').exists() and not (researcher/'cron/jobs.json').exists()
print('Research profile remains empty. No scheduler, gateway, bot, external message or public upload was created.')

## Original archive appendix

Complete original blocks and transcripts, preserved for comparison. These Markdown fences are not executable cells.

### Original block 1

````text
# SOUL.md

You are a pragmatic staff engineer.
You read existing code before writing new code.
You prefer small, safe changes over large rewrites.
You run relevant tests before saying a task is complete.
You explain trade-offs clearly and avoid unnecessary abstraction.
When debugging, first reproduce the issue, then inspect the smallest
relevant part of the codebase, then propose and verify a fix.
````

### Original block 2

````text
What did we decide last time about the authentication middleware?
````

### Original block 3

````text
Find the previous debugging session where we fixed the Docker startup issue.
````

### Original block 4

````text
This repository uses pnpm for package management. Use pnpm test to run the main test suite.
````

### Original block 5

````text
The user was debugging something today.
````

### Original block 6

````text
The current error might be caused by the database.
````

### Original block 7

````text
The backend uses FastAPI.
Tests are run with pytest.
Integration tests require the test database.
The user prefers small patches.
````

### Original block 8

````text
Backend project: FastAPI service tested with pytest. Integration tests require the test database. Prefer small, targeted patches when editing this repo.
````

### Original block 9

````text
Use `make test-api` before editing backend endpoints.
The project stores API route definitions under `src/routes/`.
Do not modify generated files under `dist/`.
The user prefers minimal diffs and explicit test output before final responses.
````

### Original block 10

````text
Prioritize primary sources, official docs, and papers over secondary summaries.
Separate confirmed facts from speculation.
Include citations for model releases, benchmark results, and pricing claims.
````

### Original block 11

````text
Use a technical, moderately formal tone.
Avoid punchline-style standalone lines.
Prefer diagrams that explain system flow or architecture.
````

### Original block 12

````text
---
name: debug-fastapi-tests
description: Use when debugging failing FastAPI backend tests.
version: 1.0.0
author: agent
---
## When to use
Use this skill when a backend test fails in the FastAPI service.
## Procedure
1. Run the focused failing test first.
2. Read the full stack trace before editing code.
3. Identify whether the failure is caused by route logic, dependency injection,
   database state, or test setup.
4. Apply the smallest safe fix.
5. Re-run the focused test.
6. Run the broader backend test command before reporting completion.
## Pitfalls
- Do not edit generated files.
- Do not skip the database fixture setup.
- Do not assume the failing test is wrong before checking the implementation.
## Verification
The focused test passes, and the broader backend test command completes successfully.
````

### Original block 13

````text
Use `pnpm test:api` for backend API tests.
Use `pnpm test:unit -- <file>` for focused unit tests.
Do not run the full end-to-end suite unless explicitly requested.
Check `tests/fixtures/` before changing mocked data.
````

### Original block 14

````text
pip install hermes-agent
````

### Original block 15

````text
hermes postinstall
````

### Original block 16

````text
curl -fsSL https://raw.githubusercontent.com/NousResearch/hermes-agent/main/scripts/install.sh | bash
````

### Original block 17

````text
source ~/.bashrc
````

### Original block 18

````text
source ~/.zshrc
````

### Original block 19

````text
hermes --help
````

### Original block 20

````text
hermes setup
````

### Original block 21

````text
hermes model
````

### Original block 22

````text
hermes
````

### Original block 23

````text
Hello. Briefly confirm that you are working and tell me what tools are available.
````

### Original block 24

````text
hermes doctor
````

### Original block 25

````text
Install Hermes
→ Configure provider
→ Start terminal chat
→ Run a simple task
→ Check diagnostics
→ Then add profiles, messaging, skills, and automation
````

### Original block 26

````text
~/.hermes/
````

### Original block 27

````text
~/.hermes/
├── config.yaml
├── .env
├── SOUL.md
├── memories/
├── skills/
├── sessions/
├── cron/
├── logs/
├── cache/
├── plugins/
└── state database files
````

### Original block 28

````text
This backend project uses pytest.
````

### Original block 29

````text
When debugging failing backend tests, first run the focused test,
inspect the stack trace, check fixtures, apply the smallest safe fix,
then rerun the focused and broader test suite.
````

### Original block 30

````text
What did we decide in the previous debugging session?
````

### Original block 31

````text
Find the session where we fixed the deployment script issue.
````

### Original block 32

````text
profiles/
├── coder/
│   ├── config.yaml
│   ├── .env
│   ├── SOUL.md
│   ├── memories/
│   ├── skills/
│   └── sessions/
│
└── researcher/
    ├── config.yaml
    ├── .env
    ├── SOUL.md
    ├── memories/
    ├── skills/
    └── sessions/
````

### Original block 33

````text
developer-assistant
````

### Original block 34

````text
A pragmatic engineering assistant that helps inspect repositories,
run tests, debug failures, write documentation, and preserve useful
project-specific procedures.
````

### Original block 35

````text
hermes profile create developer-assistant --description "Helps with coding, debugging, tests, and repository documentation."
````

### Original block 36

````text
hermes profile create developer-assistant --clone --description "Helps with coding, debugging, tests, and repository documentation."
````

### Original block 37

````text
# SOUL.md

You are a pragmatic software engineering assistant.
Your priorities are correctness, small safe changes, and clear verification.
When working with a codebase:
- Read existing code before proposing changes.
- Prefer focused edits over broad rewrites.
- Identify the relevant test command before editing.
- Run or recommend verification steps before marking work complete.
- Explain trade-offs briefly and avoid unnecessary abstraction.

When debugging:
- Reproduce the issue when possible.
- Read the full error message and stack trace.
- Inspect the smallest relevant part of the codebase first.
- Avoid assuming the test is wrong before checking the implementation.

Safety constraints:
- Do not store secrets in memory.
- Do not modify generated files unless explicitly requested.
- Do not claim a fix is verified unless tests or checks were actually run.
````

### Original block 38

````text
The user prefers small, targeted code changes over large rewrites.
The user prefers technical explanations without promotional phrasing.
When working on repositories, identify the test command before changing code.
Do not store API keys, tokens, or secrets as memory.
````

### Original block 39

````text
This project uses pnpm for package management.
The backend API routes are under src/routes/.
Use pnpm test:api for backend API tests.
Generated files under dist/ should not be edited manually.
````

### Original block 40

````text
The user was debugging a backend issue today.
The error might be related to the database.
The current branch seems messy.
````

### Original block 41

````text
Understand this whole codebase and improve it.
````

### Original block 42

````text
Inspect this repository and identify how to run the main test suite.
Do not modify files. Summarize the project structure and the commands
you found.
````

### Original block 43

````text
Find one small documentation improvement in the README.
Make the smallest useful edit, then summarize what changed.
````

### Original block 44

````text
Investigate why this focused test is failing.
First run the relevant test, then inspect the error, then propose the
smallest safe fix.
````

### Original block 45

````text
Save the confirmed test command and relevant project conventions as memory.
Do not save temporary observations or guesses.
````

### Original block 46

````text
For this repository, run pnpm test:api to execute backend API tests.
````

### Original block 47

````text
The test failed today and might be caused by middleware.
````

### Original block 48

````text
Create a skill for debugging backend API test failures in this repository.
Include when to use it, the commands to run, common pitfalls, and
verification steps.
````

### Original block 49

````text
hermes profile create coder
````

### Original block 50

````text
coder setup
coder chat
````

### Original block 51

````text
hermes profile create researcher --description "Reads papers, official docs, and technical sources, then writes concise findings."
````

### Original block 52

````text
hermes profile create work --clone
````

### Original block 53

````text
coder
researcher
designer
````

### Original block 54

````text
# SOUL.md

You are a pragmatic software engineering assistant.
Prioritize correctness, small safe changes, and verification.
Read existing code before editing.
Identify the relevant test command before changing code.
Do not claim work is complete unless verification was run or clearly recommended.
````

### Original block 55

````text
# SOUL.md
You are a careful technical research assistant.
Prioritize primary sources, official documentation, and papers.
Separate confirmed facts from speculation.
Cite claims that depend on external sources.
Summarize findings clearly without promotional language.
````

### Original block 56

````text
# SOUL.md

You are a technical diagram assistant.
Create diagrams that explain architecture, workflows, and system behavior.
Prioritize clarity, readable labels, and consistent visual structure.
Avoid decorative complexity that makes the concept harder to understand.
````

### Original block 57

````text
This repository uses pnpm.
Backend tests run with pnpm test:api.
Do not edit generated files under dist/.
The user prefers small diffs and explicit verification.
````

### Original block 58

````text
Prioritize official docs, papers, and primary sources.
Include citations for model releases, benchmark claims, pricing, and dates.
Separate confirmed facts from interpretation.
````

### Original block 59

````text
Use clean technical diagrams with readable labels.
Prefer architecture and workflow diagrams over decorative illustrations.
Keep visual style consistent across a blog series.
````

### Original block 60

````text
Which identity, memory, skills, sessions, and config should this agent use?
````

### Original block 61

````text
Which files, commands, and system resources is this agent allowed to access?
````

### Original block 62

````text
coder config set terminal.cwd /absolute/path/to/project
````

### Original block 63

````text
coder gateway start
researcher gateway start
````

### Original block 64

````text
coder/
  Purpose: repository work, debugging, tests, documentation
  Memory: project commands, conventions, generated-file rules
  Skills: debugging tests, writing docs, reviewing changes

researcher/
  Purpose: technical research and source-based summaries
  Memory: citation rules, preferred sources, topic scope
  Skills: paper summary, repo analysis, release tracking

designer/
  Purpose: diagrams and visual explanation
  Memory: visual style preferences, brand constraints, image format rules
  Skills: architecture diagrams, workflow diagrams, figure prompts
````

### Original block 65

````text
hermes profile list
hermes profile show coder
hermes profile use coder
````

### Original block 66

````text
researcher
````

### Original block 67

````text
hermes gateway setup
````

### Original block 68

````text
hermes gateway start
````

### Original block 69

````text
hermes gateway status
````

### Original block 70

````text
hermes gateway
````

### Original block 71

````text
Hello. Confirm which Hermes profile you are connected to.
````

### Original block 72

````text
Summarize what you can do in this profile and list any active tools or constraints.
````

### Original block 73

````text
@researcher_bot summarize today’s AI releases
````

### Original block 74

````text
hermes -p researcher gateway start
````

### Original block 75

````text
hermes -p researcher gateway status
````

### Original block 76

````text
hermes -p researcher gateway stop
````

### Original block 77

````text
hermes gateway start
hermes gateway status
hermes gateway stop
````

### Original block 78

````text
hermes -p coder gateway start
hermes -p researcher gateway start
hermes -p assistant gateway start
````

### Original block 79

````text
/sethome
````

### Original block 80

````text
hermes profile create coder --description "Repository work, debugging, tests, and technical documentation."
````

### Original block 81

````text
hermes -p coder chat
````

### Original block 82

````text
# SOUL.md

You are a pragmatic software engineering assistant.
Priorities:
- Understand the existing code before editing.
- Prefer small, focused patches over broad rewrites.
- Identify the relevant test or validation command before changing code.
- Explain what changed and how it was verified.
- Do not claim a task is complete unless verification was run or clearly marked as not run.
Constraints:
- Do not run destructive commands unless explicitly approved.
- Do not edit generated files unless explicitly requested.
- Do not store secrets, tokens, or private credentials in memory.
- Ask for clarification when the task depends on missing project context.
````

### Original block 83

````text
cd /path/to/project
hermes -p coder chat
````

### Original block 84

````text
Inspect this repository without modifying files.

Your task:
1. Identify the main language/framework.
2. Identify the package manager.
3. Find the main test command.
4. Summarize the top-level project structure.
5. Tell me what files or commands you inspected.
Do not edit files yet.
````

### Original block 85

````text
Verify the test command from project files before saving it as memory.
Show the file where the command is defined.
````

### Original block 86

````text
## Project memory

- This repository uses pnpm for package management.
- Main backend tests run with `pnpm test:api`.
- API route definitions are under `src/routes/`.
- Generated files under `dist/` should not be edited manually.
````

### Original block 87

````text
- The project looks complicated.
- The current branch seems messy.
- The last bug might be related to middleware.
````

### Original block 88

````text
Save only the confirmed stable project facts as memory:
- package manager
- main test command
- generated-file rule
- important source directories

Do not save temporary observations, guesses, current branch details, or the current error.
````

### Original block 89

````text
Fix the failing validation test in this repository.

Workflow:
1. First inspect the relevant test and implementation files.
2. Reproduce the failure if possible.
3. Explain the likely cause before editing.
4. Apply the smallest safe fix.
5. Re-run the focused test.
6. Summarize the diff and verification result.

Do not run broad refactors.
Do not edit unrelated files.
Do not save anything to memory unless it is a stable project convention.
````

### Original block 90

````text
Refactor the whole backend.
Improve the architecture.
Clean up this repository.
Make the code production-ready.
````

### Original block 91

````text
Inspect the backend structure and identify three small refactoring opportunities.
Do not edit files. For each opportunity, explain the risk, expected benefit,
and how it could be verified.
````

### Original block 92

````text
Create a skill for debugging backend API test failures in this repository.

The skill should include:
- when to use it
- relevant commands
- files to inspect first
- common failure categories
- what not to edit
- verification steps
````

### Original block 93

````text
---
name: debug-backend-api-tests
description: Use when debugging failing backend API tests in this repository.
version: 1.0.0
author: agent
---

## When to use
Use this skill when a backend API test fails locally or in CI.
## Procedure
1. Run the focused failing test first.
2. Read the full error output and stack trace.
3. Inspect the test file before editing implementation code.
4. Check route handlers, validation logic, fixtures, and dependency setup.
5. Apply the smallest safe fix.
6. Re-run the focused test.
7. Run the broader backend API test command before reporting completion.
## Pitfalls
- Do not edit generated files under `dist/`.
- Do not assume the test is wrong before checking the implementation.
- Do not update snapshots or fixtures unless the expected behavior changed intentionally.
## Verification
The focused test passes, and the broader backend API test command completes successfully.
````

### Original block 94

````text
Inspect the repository and improve the README setup section.

Constraints:
1. Do not change code.
2. Only update the README.
3. Keep the edit small.
4. Preserve the existing style.
5. Mention any commands you verified from project files.
````

### Original block 95

````text
Summarize the documentation change.
List the files edited.
State whether any commands were verified from project files.
Do not save this task as memory unless it revealed a stable project convention.
````

### Original block 96

````text
Create a skill for updating repository documentation.

Include:
- when to use it
- how to inspect existing docs
- how to preserve style
- how to verify commands before documenting them
- what not to change
````

### Original block 97

````text
Review the current git diff.

Focus on:
1. correctness
2. edge cases
3. tests
4. unnecessary complexity
5. possible regressions
Do not modify files.
Return findings grouped by severity.
Include the file and line when possible.
````

### Original block 98

````text
Create a code review skill for this repository.

The skill should define:
- severity levels
- review categories
- when to request tests
- how to cite files and lines
- what kinds of comments to avoid
````

### Original block 99

````text
Changed:
- src/routes/users.ts

Summary:
- Added validation for missing email before creating a user.

Verification:
- Ran pnpm test:api -- users.test.ts
- Result: passed

Not verified:
- Full end-to-end suite was not run.

Next step:
- Run pnpm test:e2e if you want broader coverage.
````

### Original block 100

````text
Done. The issue should be fixed.
````

### Original block 101

````text
git status
````

### Original block 102

````text
git diff
git status
````

### Original block 103

````text
Review your own diff.

Check for:
1. unrelated changes
2. unnecessary complexity
3. missing verification
4. files that should not have been edited
Do not modify files during this review.
````

### Original block 104

````text
The repository uses `pnpm test:api` for backend tests.
````

### Original block 105

````text
How to debug backend API test failures in this repository.
````

### Original block 106

````text
This specific test failed today because of a local branch change.
````

### Original block 107

````text
Review what you learned from this task.

Classify each item as:
1. memory
2. skill update
3. session-only detail
Only save items that are stable and reusable.
Do not save temporary errors, branch-specific observations, or guesses.
````

### Original block 108

````text
Inspect this repository and identify the test commands.
Update the README setup section using commands verified from project files.
Debug this focused failing test and apply the smallest safe fix.
Review this git diff for correctness and missing tests.
Create a skill for the repeated debugging workflow we just used.
````

### Original block 109

````text
Improve the whole codebase.
Refactor the architecture.
Make this production-ready.
Fix all bugs.
Clean everything up.
````

### Original block 110

````text
hermes profile create researcher --description "Researches AI, ML, software tools, papers, and technical sources."
````

### Original block 111

````text
hermes profile create researcher --clone --description "Researches AI, ML, software tools, papers, and technical sources."
````

### Original block 112

````text
hermes profile create researcher --clone-from default --description "Researches AI, ML, software tools, papers, and technical sources."
````

### Original block 113

````text
hermes -p researcher chat
````

### Original block 114

````text
# SOUL.md

You are a careful technical research assistant for AI, machine learning,
software engineering, and developer tools.

Your priorities:
- Prefer primary sources: official docs, papers, release notes, GitHub repos,
  technical blogs from authors, and standards documents.
- Separate confirmed facts from interpretation.
- Cite claims that depend on external sources.
- Do not present weak evidence as settled.
- Summarize what changed, why it matters, and what remains uncertain.
- Keep outputs concise, structured, and useful for technical readers.

When researching:
- Identify the source type before relying on it.
- Prefer recent sources for fast-moving topics.
- Cross-check important claims when possible.
- Flag when information may be incomplete, outdated, or based on limited evidence.
- Avoid copying long passages from sources.

Do not fabricate citations, dates, benchmark results, prices, or technical claims.
````

### Original block 115

````text
# USER.md

- The user prefers technical, moderately formal explanations.
- The user dislikes vague promotional language and punchline-style phrasing.
- The user prefers source-grounded research with citations for factual claims.
- The user wants uncertainty clearly separated from confirmed facts.
- The user is interested in AI agents, LLM systems, RAG, model training,
  agent memory, evaluation, and open-source AI tools.
````

### Original block 116

````text
# MEMORY.md

## Research workflow memory
- Prefer primary sources over secondary summaries.
- For model releases, check official announcements, documentation, model cards,
  GitHub repos, and papers when available.
- For benchmark claims, record the benchmark name, model version, date,
  source, and any stated evaluation conditions.
- For pricing, API limits, or product availability, treat information as
  time-sensitive and verify from official sources.
- For papers, summarize the problem, method, main results, limitations,
  and practical relevance.
````

### Original block 117

````text
Prefer papers and official docs over social media summaries.
For model-release claims, verify date, version, license, and source.
Separate confirmed results from author claims or community interpretation.
````

### Original block 118

````text
The user read a paper today.
This model seems interesting.
There was some drama on X about a benchmark.
````

### Original block 119

````text
Research the latest Hermes Agent documentation and summarize the current
capabilities relevant to developers.

Scope:
1. Installation and setup.
2. Profiles.
3. Memory.
4. Skills.
5. Messaging.
6. Scheduled jobs.

Source policy:
- Prefer official Hermes documentation and GitHub sources.
- Do not rely on unsourced social posts.
- Cite each factual claim that depends on external sources.

Output format:
- Key findings
- Practical implications for developers
- Limitations or unclear areas
- Links/sources used
````

### Original block 120

````text
Tell me everything about Hermes Agent.
````

### Original block 121

````text
## Summary

Short description of what was released.
## What changed
- Model/version:
- Release date:
- Provider/lab:
- License or access:
- Context length:
- Tooling/API changes:
- Notable benchmark or capability claims:
## Why it matters
Practical implications for developers and AI practitioners.
## Limitations and uncertainty
What is not confirmed, what needs testing, or what depends on vendor claims.
## Sources
Cited sources used in the answer.
````

### Original block 122

````text
## Problem

What problem the paper is addressing.
## Method
The main technical idea.
## Results
Main reported findings and evaluation setup.
## Limitations
What the paper does not solve or what assumptions it makes.
## Practical relevance
Why developers or AI practitioners should care.
## Sources
Paper link and any related official resources.
````

### Original block 123

````text
## What the project does

Short technical description.
## Why it is relevant
What problem it solves and who might use it.
## Repository health
Stars, activity, issues, license, installation clarity, docs quality.
## Technical notes
Architecture, dependencies, supported platforms, APIs, limitations.
## Risks or open questions
Maintenance risk, unclear license, missing tests, security concerns.
## Sources
Repository and related docs.
````

### Original block 124

````text
Create a skill for summarizing AI research papers.

The skill should include:
- when to use it
- source requirements
- required fields
- how to separate claims from interpretation
- how to handle missing information
- the final output format
````

### Original block 125

````text
---
name: summarize-ai-paper
description: Use when summarizing AI, ML, or agent-systems research papers.
version: 1.0.0
author: agent
---
## When to use
Use this skill when the user asks for a structured summary of a technical paper.
## Procedure
1. Identify the paper title, authors, venue or preprint status, and date.
2. Read the abstract, introduction, method, experiments, and limitations.
3. Extract the problem, method, evaluation setup, main results, and limitations.
4. Separate reported claims from your own interpretation.
5. Avoid overstating benchmark results.
6. Include citations or links for the paper and any related official resources.
## Output format
- Problem
- Method
- Results
- Limitations
- Practical relevance
- Sources
## Pitfalls
- Do not summarize only from social posts.
- Do not treat author claims as independently verified results.
- Do not omit limitations.
````

### Original block 126

````text
Summarize this paper for developers and AI practitioners.

Focus on:
1. The problem the paper addresses.
2. The core technical idea.
3. The evaluation setup.
4. The main reported results.
5. The limitations.
6. What this changes in practice.
Separate author claims from your own interpretation.
Do not overstate the result.
````

### Original block 127

````text
Also check whether the authors provide code, model weights, datasets,
or reproducibility notes. Include the status in the summary.
````

### Original block 128

````text
Research this model release and prepare a technical brief.

Check:
1. Official announcement.
2. Documentation or model card.
3. GitHub repository, if available.
4. License or usage terms.
5. API or deployment details.
6. Benchmark claims and evaluation setup.
7. Practical implications for developers.
Do not rely only on social media summaries.
Flag anything that is unclear or not independently verified.
````

### Original block 129

````text
Compare Tool A and Tool B for developers building production AI agents.

Compare:
1. Core purpose.
2. Architecture.
3. Setup complexity.
4. Supported integrations.
5. Memory/state handling.
6. Tool execution model.
7. Deployment options.
8. Strengths and limitations.
9. Best fit use cases.
Use primary sources where possible.
Return a balanced technical comparison, not a promotional summary.
````

### Original block 130

````text
## Summary table

## Tool A: strengths and limitations
## Tool B: strengths and limitations
## Where Tool A fits better
## Where Tool B fits better
## Open questions
## Sources
````

### Original block 131

````text
Review what you learned from this research task.

Classify each item as:
1. user preference
2. reusable research workflow
3. topic-specific fact
4. session-only detail
Only save stable preferences or reusable workflows.
Do not save temporary news, guesses, or unverified claims as memory.
````

### Original block 132

````text
Find the session where we compared agent frameworks last week.
````

### Original block 133

````text
What sources did we use when summarizing the GEPA paper?
````

### Original block 134

````text
Summarize this paper for AI practitioners using a fixed template.
Compare these two tools using official docs and GitHub sources.
Research this model release and separate confirmed facts from claims.
Prepare a technical brief on this new agent framework.
Track what changed in this project since the last release.
````

### Original block 135

````text
Tell me everything about this topic.
Find the best AI tool.
Summarize what people are saying.
What is the truth about this controversy?
````

### Original block 136

````text
hermes -p researcher chat
````

### Original block 137

````text
Create a concise technical briefing on AI agent and open-source LLM updates from the past 24 hours.

Requirements:
1. Prefer official announcements, papers, GitHub repositories, and primary sources.
2. Include 3 to 5 important updates.
3. For each update, include the source, what changed, and why it matters.
4. Separate confirmed facts from interpretation.
5. Keep the briefing concise and technical.
````

### Original block 138

````text
Every weekday at 8am, create a concise technical briefing on AI agents,
open-source LLMs, RAG, and agent memory.

Use official announcements, papers, GitHub repositories, and primary sources
where possible. Include 3 to 5 important updates. For each update, include:
1. What changed.
2. Why it matters for developers or AI practitioners.
3. Source link.
4. Any uncertainty or missing information.
Deliver the result to Telegram.
````

### Original block 139

````text
hermes -p researcher cron list
````

### Original block 140

````text
/cron add "0 8 * * 1-5" "Create a concise weekday technical briefing on AI agents, open-source LLMs, RAG, and agent memory.
Requirements:
- Prefer official announcements, papers, GitHub repositories, and primary sources.
- Include 3 to 5 important updates.
- For each update, include what changed, why it matters, the source, and any uncertainty.
- Keep the output concise and technical.
- Do not include weak social media claims unless they are clearly labeled as unverified.
Deliver to Telegram."
````

### Original block 141

````text
Do my usual research digest.
````

### Original block 142

````text
hermes -p researcher cron list
````

### Original block 143

````text
hermes -p coder cron list
````

### Original block 144

````text
hermes -p researcher gateway start
````

### Original block 145

````text
/sethome
````

### Original block 146

````text
hermes -p researcher gateway
````

### Original block 147

````text
hermes -p researcher gateway start
````

### Original block 148

````text
hermes -p researcher gateway status
````

### Original block 149

````text
hermes -p researcher cron list
````

### Original block 150

````text
model-release-brief
````

### Original block 151

````text
/cron add "0 8 * * 1-5" "Prepare a weekday model-release brief for AI practitioners. Use the model-release-brief skill. Focus on official releases and major open-source updates from the past 24 hours. Deliver to Telegram."
````

### Original block 152

````text
Check whether the target repository has a new release since the last run.

If there is a meaningful new release, summarize:
1. Version.
2. Release date.
3. Important changes.
4. Source link.
If there is no meaningful change, respond with only:
[SILENT]
````

### Original block 153

````text
/cron add "0 8 * * 1-5" "Create a weekday AI research briefing for developers and AI practitioners.
Scope:
- AI agents
- open-source LLMs
- RAG
- agent memory
- model evaluation
Source policy:
- Prefer official announcements, papers, GitHub repos, and documentation.
- Include source links.
- Avoid unverified social media claims unless clearly labeled.
Output:
- 3 to 5 important updates
- what changed
- why it matters
- uncertainty or missing information
Deliver to Telegram."
````

### Original block 154

````text
/cron add "0 17 * * 5" "Create a weekly repository health summary.Inspect the current repository and summarize:
1. Recent commits.
2. Test status if available.
3. Open TODO or FIXME comments.
4. Documentation areas that may need updates.
5. Risky or unclear changes.
Do not modify files.
Return a concise report with suggested next actions."
````

### Original block 155

````text
/cron add "0 10 * * 1" "Generate 5 technical blog ideas based on recent AI agent and LLM infrastructure developments.For each idea include:
1. Working title.
2. One-sentence angle.
3. Why developers should care.
4. Suggested diagram.
5. Source direction to investigate.
Do not draft full posts."
````

### Original block 156

````text
hermes -p researcher cron list
hermes -p researcher cron run <job_id>
hermes -p researcher cron edit <job_id>
hermes -p researcher logs
hermes -p researcher skills list
````

### Original block 157

````text
date
hermes -p researcher cron list
````

### Original quotation/output 1

````text
This limits how useful an agent can become in long-running workflows.
````

### Original quotation/output 2

````text
The interaction between these layers is where the system becomes useful.
````

### Original quotation/output 3

````text
The practical value of Hermes comes from what happens before, during, and after this loop.
````

### Original quotation/output 4

````text
This is different from memory.
````

### Original quotation/output 5

````text
It is also worth noting that SOUL.md should be reviewed over time. If the agent repeatedly behaves in a way that does not match your expectations, the issue may not only be the model. The identity file may be too vague, too broad, or missing an important constraint. Updating SOUL.md is often a more direct fix than repeatedly correcting the agent in individual sessions.
````

### Original quotation/output 6

````text
This layered design is important because different types of information need different storage behavior.
````

### Original quotation/output 7

````text
A good skill should be easy for the agent to recognize, specific enough to execute, and clear about verification. Weak skills can create the opposite effect: they may load at the wrong time, add unnecessary context, or push the agent toward outdated procedures.
````

### Original quotation/output 8

````text
Hermes addresses this through a maintenance mechanism called the Curator.
````

### Original quotation/output 9

````text
The Curator helps reduce this risk by treating the skill library as something that needs lifecycle management.
````

### Original quotation/output 10

````text
Without a maintenance process, the agent may keep treating both types of skills as equally relevant.
````

### Original quotation/output 11

````text
This is the reason offline optimization matters.
````

### Original quotation/output 12

````text
The difference between runtime learning and offline optimization is important.
````

### Original quotation/output 13

````text
Hermes is installed.
A model provider is configured.
A terminal chat works successfully.
````

### Original quotation/output 14

````text
SOUL.md → how the agent should behave
memory → what the agent should remember
skills → how the agent should perform repeated workflows
````

### Original quotation/output 15

````text
This is where messaging gateways become important.
````

### Original quotation/output 16

````text
Do not connect every profile to messaging on day one. Start with one profile and verify that it works.
````

### Original quotation/output 17

````text
This is one of the main advantages of using Hermes as a coding assistant. Once the test command and project conventions are saved, future sessions do not need to rediscover them from scratch.
````

### Original quotation/output 18

````text
The weak tasks are not impossible, but they are too broad for early agent use. They should be decomposed into inspection, planning, implementation, and verification steps.
````

### Original quotation/output 19

````text
Cron changes the operating model.
````

### Original quotation/output 20

````text
The schedule controls when the job runs. The prompt tells Hermes what to do. The active profile controls the agent identity, memory, skills, tools, and configuration. The delivery target controls where the result should go.
````

### Original quotation/output 21

````text
Only schedule the task after the manual version works.
````

### Original quotation/output 22

````text
short cron prompt + attached skill + profile memory = consistent recurring output
````

### Original quotation/output 23

````text
schedule
→ run script
→ stdout delivered to messaging platform
→ no model call
````

### Original quotation/output 24

````text
Scheduled jobs should produce reviewable outputs before they produce irreversible actions.
````

### Original quotation/output 25

````text
The best way to start is narrow.
````

### Original quotation/output 26

````text
This distinction matters especially for coding and automation workflows.
````